# Imports and Setup

In [1]:
!pip show spicelib
# !pip install spicelib # install if needed
# !pip install --upgrade ipykernel jedi


Name: spicelib
Version: 1.4.6
Summary: A set of tools to Automate Spice simulations
Home-page: 
Author: Nuno Brum
Author-email: nuno.brum@gmail.com
License: GPL-3.0
Location: /foss/designs/eda/.venv/lib/python3.12/site-packages
Requires: matplotlib, numpy, psutil, scipy
Required-by: 


In [2]:
from spicelib import SpiceEditor, SimRunner, RawRead
from spicelib.simulators import ngspice_simulator 

import logging
import os

from pathlib import Path


In [3]:
import logging

# Create a logger
logger = logging.getLogger("notebook_logger")
logger.setLevel(logging.DEBUG)  # Set lowest level you want to capture (DEBUG, INFO, WARNING...)

# Create a console handler (for notebook output)
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.DEBUG)  # You can choose INFO if DEBUG is too noisy
logger.propagate = False  # stop passing logs to root logger

# Create a formatter
formatter = logging.Formatter(
    fmt="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
console_handler.setFormatter(formatter)

# Add the handler to the logger (avoid duplicates)
if not logger.handlers:
    logger.addHandler(console_handler)

# Example usage
logger.info("Logger initialized.")

23:10:25 [INFO] Logger initialized.


## Simulation Config

In [4]:
PATH_TO_NGSPICE = Path("/foss/tools/bin/ngspice")

PROJECT_NAME    = "tia-bpf-1"
SCHEMATIC_NAME  = "tb_ac"
SIZER_NAME      = "simple"

OUTPUT_DIR      = Path(f"./runs/{SIZER_NAME}/{PROJECT_NAME}")
INITIAL_NETLIST = Path(f"../../{PROJECT_NAME}/netlist/{SCHEMATIC_NAME}.spice")

os.makedirs(OUTPUT_DIR, exist_ok=True)
if not INITIAL_NETLIST.exists():
    raise FileNotFoundError(f"Initial netlist not found: {INITIAL_NETLIST}")


logger.info(f"Using ngspice from {PATH_TO_NGSPICE}")
logger.info(f"project: {PROJECT_NAME}, schematic: {SCHEMATIC_NAME}")

23:10:25 [INFO] Using ngspice from /foss/tools/bin/ngspice
23:10:25 [INFO] project: tia-bpf-1, schematic: tb_ac


# SpiceLib

## Create a runner

In [ ]:
runner = SimRunner(
    simulator=ngspice_simulator.NGspiceSimulator.create_from(path_to_exe=PATH_TO_NGSPICE), 
    # simulator=ngspice_simulator.NGspiceSimulator, 
    output_folder=OUTPUT_DIR,
    cwd=OUTPUT_DIR,
    
    )

runner.cwd = Path("./")

## Load the initial (unsized) TB schematic

In [6]:
# Create a SpiceEditor Instance
editor = SpiceEditor(netlist_file=INITIAL_NETLIST)

# Nodes
nodes = editor.get_all_nodes()
logger.info(f"Nodes in the netlist:\n{nodes}")

# Parameters
params = editor.get_all_parameter_names()
tb_params  = [(param, editor.get_parameter(param)) for param in params if not "X_DUT" in param]
dut_params = [(param, editor.get_parameter(param)) for param in params if "X_DUT" in param]
logger.info(f"Testbench parameters:\n{tb_params}")
logger.info(f"DUT parameters:\n{dut_params}")



23:10:25 [INFO] Nodes in the netlist:
['VSS', 'GND', 'VDD', 'Vop', 'Von', 'In', 'Ip', 'Vbias']
23:10:25 [INFO] Testbench parameters:
[('CLOAD', '1p'), ('ICM', '0.5e-6'), ('IDD', '1.8e-6'), ('VBIAS', '1.0'), ('VDD', '6.0')]
23:10:25 [INFO] DUT parameters:
[('X_DUT_CDD', '1n'), ('X_DUT_LDD', '2n'), ('X_DUT_M1_2_L', '0.40u'), ('X_DUT_M1_2_NF', '2'), ('X_DUT_M1_2_W', '0.22u'), ('X_DUT_R4', '1k'), ('X_DUT_RS', '1k')]


### Run a sanity
Make sure the schematic works

In [7]:
runtask = runner.run_now(
    netlist=INITIAL_NETLIST,
    exe_log=True )
runtask

RunTask #1:Simulation Aborted. Time elapsed: 00.0075 secs


(None, PosixPath('runs/simple/tia-bpf-1/tb_ac_1.fail'))

In [8]:
# editor.prepare_for_simulator(ngspice_simulator.Simulator)


In [9]:
editor.set_parameters(VIN=2)
editor.save_netlist(f"{OUTPUT_DIR}/saved.spice")

In [10]:
print(OUTPUT_DIR)

runs/simple/tia-bpf-1


### saving and reading RAW files

In [11]:
from typing import List
global_raw_files : List[RawRead] = []
def processing_data(raw_filename, log_filename):
    '''This is a call back function that just prints the filenames'''
    print("Simulation Raw file is %s. The log is %s" % (raw_filename, log_filename))
    # Other code below either using ltsteps.py or raw_read.py
    # log_info = LTSpiceLogReader(log_filename)
    # log_info.read_measures()
    # rise, measures = log_info.dataset["rise_time"]
    global_raw_files.append(RawRead(raw_filename=raw_filename))
    return global_raw_files[raw_filename]

In [12]:
runtask = runner.run(
    netlist=INITIAL_NETLIST,
    callback=processing_data,
    # run_filename="runfile.spice",
    exe_log=True )
runtask

<RunTask(RunTask#2, started 281472179171712)>

In [13]:
runtask.get_results()

In [14]:
runner.sim_info()

{1: {'netlist_file': PosixPath('runs/simple/tia-bpf-1/tb_ac_1.spice'),
  'raw_file': None,
  'log_file': PosixPath('runs/simple/tia-bpf-1/tb_ac_1.fail'),
  'retcode': 1,
  'exception_text': None,
  'callback_return': None,
  'start_time': 1757538625.8563902,
  'stop_time': 1757538625.9320774}}

In [15]:
raw = global_raw_files[0]
raw.get_trace_names()

RunTask #2:Simulation Aborted. Time elapsed: 00.0136 secs


IndexError: list index out of range

In [ ]:
raw.get_plot_names()

In [ ]:
raw.get_plot_name()

In [ ]:
for plot in raw.plots:
    print(plot.get_plot_name())
    print(len(plot.get_trace("v(vout)").get_wave()))
    

In [ ]:
raw.get_trace("v(vout)").get_wave()

## Batch Runner

In [ ]:
# TODO

# Optimizer Logic

In [ ]:
# TODO